# FORESIGHT — Data Quality & Validation

## Demand & Inventory Intelligence

This notebook performs data-quality assessment and validation
for the four core FORESIGHT datasets:

1. sales_daily
2. sku_master
3. calendar
4. inventory_snapshots

The objective is to validate data completeness, consistency,
validity, and cross-table relationships before exploratory
data analysis and demand forecasting.

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [40]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"

DATA_DIR

WindowsPath('c:/Users/shiva/OneDrive/Desktop/Company/foresight-demand-inventory-intelligence/data/processed')

In [41]:
sales = pd.read_csv(
    DATA_DIR / "sales_daily.csv"
)

sku = pd.read_csv(
    DATA_DIR / "sku_master.csv"
)

calendar = pd.read_csv(
    DATA_DIR / "calendar.csv"
)

inventory = pd.read_csv(
    DATA_DIR / "inventory_snapshots.csv"
)

In [42]:
print("Sales:", sales.shape)
print("SKU Master:", sku.shape)
print("Calendar:", calendar.shape)
print("Inventory:", inventory.shape)

Sales: (4143430, 6)
SKU Master: (5000, 6)
Calendar: (1461, 6)
Inventory: (24872, 6)


In [43]:
dataset_summary = pd.DataFrame({
    "Dataset": [
        "sales_daily",
        "sku_master",
        "calendar",
        "inventory_snapshots"
    ],
    "Rows": [
        len(sales),
        len(sku),
        len(calendar),
        len(inventory)
    ],
    "Columns": [
        sales.shape[1],
        sku.shape[1],
        calendar.shape[1],
        inventory.shape[1]
    ]
})

dataset_summary

,Dataset,Rows,Columns
0,sales_daily,4143430,6
1,sku_master,5000,6
2,calendar,1461,6
3,inventory_snapshots,24872,6


In [44]:
required_columns = {
    "sales_daily": [
        "date",
        "sku_id",
        "units_sold",
        "revenue",
        "unit_price",
        "promo_flag"
    ],

    "sku_master": [
        "sku_id",
        "category",
        "subcategory",
        "launch_date",
        "unit_cost",
        "list_price"
    ],

    "calendar": [
        "date",
        "week",
        "month",
        "season",
        "is_holiday",
        "promo_event"
    ],

    "inventory_snapshots": [
        "date",
        "sku_id",
        "on_hand_units",
        "on_order_units",
        "lead_time_days",
        "reorder_point"
    ]
}

In [45]:
datasets = {
    "sales_daily": sales,
    "sku_master": sku,
    "calendar": calendar,
    "inventory_snapshots": inventory
}

for name, df in datasets.items():
    missing = [
        col for col in required_columns[name]
        if col not in df.columns
    ]

    print(f"{name}:")
    print("Missing required columns:", missing)
    print()

sales_daily:
Missing required columns: []

sku_master:
Missing required columns: []

calendar:
Missing required columns: []

inventory_snapshots:
Missing required columns: []



In [46]:
for name, df in datasets.items():

    print("=" * 60)
    print(name)

    display(
        pd.DataFrame({
            "column": df.columns,
            "dtype": df.dtypes.astype(str)
        })
    )

sales_daily


,column,dtype
date,date,str
sku_id,sku_id,str
units_sold,units_sold,int64
revenue,revenue,float64
unit_price,unit_price,float64
promo_flag,promo_flag,int64


sku_master


,column,dtype
sku_id,sku_id,str
category,category,str
subcategory,subcategory,str
launch_date,launch_date,str
unit_cost,unit_cost,float64
list_price,list_price,float64


calendar


,column,dtype
date,date,str
week,week,int64
month,month,int64
season,season,str
is_holiday,is_holiday,int64
promo_event,promo_event,str


inventory_snapshots


,column,dtype
date,date,str
sku_id,sku_id,str
on_hand_units,on_hand_units,int64
on_order_units,on_order_units,int64
lead_time_days,lead_time_days,int64
reorder_point,reorder_point,int64


In [47]:
sales["date"] = pd.to_datetime(
    sales["date"],
    errors="coerce"
)

sku["launch_date"] = pd.to_datetime(
    sku["launch_date"],
    errors="coerce"
)

calendar["date"] = pd.to_datetime(
    calendar["date"],
    errors="coerce"
)

inventory["date"] = pd.to_datetime(
    inventory["date"],
    errors="coerce"
)

In [48]:
for name, df in datasets.items():

    print("=" * 60)
    print(name)

    missing = df.isna().sum()

    display(
        missing[
            missing > 0
        ].sort_values(ascending=False)
    )

sales_daily


Series([], dtype: int64)

sku_master


Series([], dtype: int64)

calendar


Series([], dtype: int64)

inventory_snapshots


Series([], dtype: int64)

In [49]:
duplicate_checks = {
    "sales_daily": sales.duplicated(
        ["date", "sku_id"]
    ).sum(),

    "sku_master": sku.duplicated(
        ["sku_id"]
    ).sum(),

    "calendar": calendar.duplicated(
        ["date"]
    ).sum(),

    "inventory_snapshots": inventory.duplicated(
        ["date", "sku_id"]
    ).sum()
}

duplicate_checks

{'sales_daily': np.int64(0),
 'sku_master': np.int64(0),
 'calendar': np.int64(0),
 'inventory_snapshots': np.int64(0)}

In [50]:
print(
    "Negative units:",
    (sales["units_sold"] < 0).sum()
)

print(
    "Negative revenue:",
    (sales["revenue"] < 0).sum()
)

print(
    "Negative price:",
    (sales["unit_price"] < 0).sum()
)

print(
    "Invalid promo flags:",
    (~sales["promo_flag"].isin([0, 1])).sum()
)

Negative units: 0
Negative revenue: 0
Negative price: 0
Invalid promo flags: 0


In [51]:
print(
    "Negative unit cost:",
    (sku["unit_cost"] < 0).sum()
)

print(
    "Negative list price:",
    (sku["list_price"] < 0).sum()
)

Negative unit cost: 0
Negative list price: 0


In [52]:
print(
    "Invalid holiday flags:",
    (~calendar["is_holiday"].isin([0, 1])).sum()
)

print(
    "Invalid months:",
    (~calendar["month"].between(1, 12)).sum()
)

print(
    "Invalid weeks:",
    (~calendar["week"].between(1, 53)).sum()
)

Invalid holiday flags: 0
Invalid months: 0
Invalid weeks: 0


In [53]:
inventory_numeric = [
    "on_hand_units",
    "on_order_units",
    "lead_time_days",
    "reorder_point"
]

for column in inventory_numeric:

    print(
        f"Negative {column}:",
        (inventory[column] < 0).sum()
    )

Negative on_hand_units: 0
Negative on_order_units: 0
Negative lead_time_days: 0
Negative reorder_point: 0


In [54]:
sales_skus = set(
    sales["sku_id"].astype(str)
)

master_skus = set(
    sku["sku_id"].astype(str)
)

missing_sales_skus = sales_skus - master_skus

print(
    "Sales SKUs missing from SKU master:",
    len(missing_sales_skus)
)

Sales SKUs missing from SKU master: 0


In [55]:
inventory_skus = set(
    inventory["sku_id"].astype(str)
)

missing_inventory_skus = inventory_skus - master_skus

print(
    "Inventory SKUs missing from SKU master:",
    len(missing_inventory_skus)
)

Inventory SKUs missing from SKU master: 0


In [56]:
skus_without_inventory = master_skus - inventory_skus

print(
    "SKU master SKUs without inventory:",
    len(skus_without_inventory)
)

SKU master SKUs without inventory: 0


In [57]:
sales_dates = set(
    sales["date"].dropna()
)

calendar_dates = set(
    calendar["date"].dropna()
)

missing_calendar_dates = sales_dates - calendar_dates

print(
    "Sales dates missing from calendar:",
    len(missing_calendar_dates)
)

Sales dates missing from calendar: 0


## Data Quality Findings

### Overall Result

All four FORESIGHT datasets passed the automated technical
validation checks.

### Key Findings

1. sales_daily contains 4,143,561 records with no missing
   required values or duplicate date-SKU records.

2. sku_master contains 5,000 unique SKUs with no missing
   required values or duplicate SKU IDs.

3. calendar contains 1,461 unique dates with all required
   fields populated.

4. Dates without an active promotion are represented as
   "No Promotion" in promo_event.

5. inventory_snapshots contains inventory records covering
   all 5,000 SKUs after the documented zero-inventory
   completion assumption.

6. No negative values were detected in the validated
   numeric fields.

7. Sales dates have complete coverage in the calendar table.

## Assumptions and Limitations

- Inventory coverage is incomplete for 519 SKUs.
- Any derived inventory planning fields must be treated as
  assumptions rather than observed supplier data.
- Calendar promotion events are derived from the supplied
  promotions date ranges.
- "No Promotion" represents dates without an active promotion.## Assumptions & Limitations

1. The original inventory source contains 4,495 unique SKUs,
   while sku_master contains 5,000 SKUs.

2. For the 505 SKUs absent from the original inventory source,
   inventory values were set to zero at the latest snapshot
   date. This is a modelling assumption and not an observed
   inventory value.

3. lead_time_days uses a documented 7-day planning assumption
   because supplier lead-time information was not available
   in the source.

4. on_order_units is a derived planning value rather than
   observed purchase-order data.

5. promo_event is derived from the supplied promotions.csv
   using promotion start and end dates.

6. "No Promotion" indicates that no supplied promotion was
   active on that date.

7. Data-quality validation confirms structural and value
   consistency but does not guarantee that all modelling
   assumptions are statistically optimal.
- Data-quality validation does not imply that the data is
  statistically suitable for every modelling approach;
  additional EDA and time-series validation are required.

## D1 Conclusion

The four core FORESIGHT datasets have been validated against
the required project schema.

The automated quality checks confirm that the datasets contain
the required columns, have no missing required values, contain
no duplicate business keys, and pass the defined numeric and
cross-table validation rules.

The processed datasets are ready for Exploratory Data Analysis.

The documented inventory completion and other derived fields
will be treated as explicit project assumptions throughout
the modelling workflow.